# Setup

In [ ]:
import re
import requests
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas import DataFrame
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import statsmodels.api as sm
from nltk.corpus import stopwords
from gensim import corpora
import gensim
from empath import Empath
import json
import os
from math import ceil
from pathlib import Path
import mpltern
import torch
from dotenv import load_dotenv
from nltk.corpus import sentiwordnet as swn
from nltk.stem import WordNetLemmatizer
from nltk.wsd import lesk
from transformers import AutoModel, AutoTokenizer


In [ ]:
nltk.download('brown')
nltk.download('wordnet_ic')
nltk.download('gutenberg')
nltk.download('punkt_tab')
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("sentiwordnet")


# Shared functions

In [ ]:
characters = {
    "Elizabeth": ["Elizabeth", "Lizzy", "Eliza"],
    "Darcy": ["Darcy"],
    "Jane": ["Jane"],
    "Bingley": ["Bingley"],
    "Wickham": ["Wickham"],
    "Lydia": ["Lydia"],
    "Mrs. Bennet": ["Mrs. Bennet"],
    "Mr. Bennet": ["Mr. Bennet"],
    "Collins": ["Collins"],
    "Charlotte": ["Charlotte"],
    "Lady Catherine": ["Lady Catherine"],
    "Kitty": ["Kitty"],
    "Mary": ["Mary"],
}


def getBookText():
    url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
    response = requests.get(url)
    return response.text


def getSpeech():
    text = getBookText().replace("\r\n", "\n")
    paragraphs = text.split("\n\n")

    speech = {name: [] for name in characters}
    for p in paragraphs:
        p = " ".join(p.split())
        quotes = re.findall(r'[“"](.*?)[”"]', p)
        if not quotes:
            continue
        narration = re.sub(r'[“"].*?[”"]', " ", p)
        for name, aliases in characters.items():
            pattern = r"\b(" + "|".join(aliases) + r")\b"
            if re.search(pattern, narration):
                speech[name].extend(quotes)
                break
    return speech


def getSpeechAsTokens():
    tokenDict = dict()
    speech = getSpeech()
    for person, quotes in speech.items():
        quotesString = ' '.join(quotes)
        quotesString = quotesString.lower()
        tokens = nltk.word_tokenize(quotesString)
        tokenDict[person] = tokens
    return tokenDict


def getBookTokens():
    text = getBookText()
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    return tokens


def printMatrix(matrix, rowLabels=None, columnLabels=None, decimalCount=2, labels=True):
    dataFrame = DataFrame(
        matrix,
        columns=columnLabels,
        index=rowLabels
    )
    print(dataFrame.round(decimalCount).to_string(header=labels, index=labels))


def printMatrixTransposed(matrix, rowLabels=None, columnLabels=None, decimalCount=2, labels=True):
    dataFrame = DataFrame(
        matrix,
        columns=columnLabels,
        index=rowLabels
    )
    print(dataFrame.T.round(decimalCount).to_string(header=labels, index=labels))


def filterTokenListTask5And6(tokenList):
    generalStopWords = stopwords.words('english')
    manualStopWords = ['must', 'one', 'shall', 'could', 'would', 'may', 'think', 'must', 'us', 'much',
                       'well', 'know', 'never', 'see', 'good', 'oh', 'ever', 'always', 'make', 'sure', 'dear', 'indeed', 'better',
                       'man', 'nothing', 'like', 'young', 'miss', 'give', 'say', 'hope', 'really', 'certainly',
                       'way', 'manner', 'go', 'little', 'let', 'time', 'ask', 'great', 'last', 'upon', 'often', 'quite', 'soon', 'come', 'tell']

    filteredTokens = []
    for token in tokenList:
        # Tokens have to be alphanumeric
        if not token.isalnum():
            continue
        # Tokens cannot be stopwords
        if token in generalStopWords:
            continue
        if token in manualStopWords:
            continue
        filteredTokens.append(token)

    return filteredTokens


def filterTokenListTask7(tokenList):
    generalStopWords = stopwords.words('english')
    manualStopWords = ['must', 'one', 'shall', 'could', 'would', 'may', 'think', 'must',
                       'us', 'much', 'well', 'know', 'never', 'see', 'good', 'oh', 'ever', 'always', 'make', 'sure', 'dear', 'indeed']

    filteredTokens = []
    for token in tokenList:
        # Tokens have to be alphanumeric
        if not token.isalnum():
            continue
        # Tokens cannot be stopwords
        if token in generalStopWords:
            continue
        if token in manualStopWords:
            continue
        filteredTokens.append(token)

    return filteredTokens


def getSpeechAsFilteredTokens(filterTokenList):
    speechAsTokens = getSpeechAsTokens()

    speechAsFilteredTokens = dict()
    for person, tokens in speechAsTokens.items():
        speechAsFilteredTokens[person] = filterTokenList(tokens)

    return speechAsFilteredTokens


In [ ]:
dataframes = {}
for name, quotes in getSpeech().items():
    if quotes:
        dataframes[name] = pd.DataFrame({"text": quotes})


# Task 1

In [ ]:
def task1():
    text = " ".join(getBookText().split())
    sentences = nltk.sent_tokenize(text)

    results = []
    for name, aliases in characters.items():
        pattern = r"\b(" + "|".join(aliases) + r")\b"
        frequency = len(re.findall(pattern, text))
        citations = sum(1 for s in sentences if re.search(pattern, s))
        results.append((name, frequency, citations))

    results.sort(key=lambda r: r[1], reverse=True)

    print(f"{'Rank':<6}{'Character':<16}{'Frequency':<11}{'Citations'}")
    for i, (name, frequency, citations) in enumerate(results, 1):
        print(f"{i:<6}{name:<16}{frequency:<11}{citations}")


In [ ]:
task1()


# Task 2

In [ ]:
def task2():
    speech = getSpeech()

    # One dataframe per character + statistics
    rows = []
    for name, quotes in speech.items():
        df = pd.DataFrame({"text": quotes})
        tokens = nltk.word_tokenize(" ".join(df["text"]))
        words = [w.lower() for w in tokens if w.isalpha()]
        vocabulary = set(words)
        tags = [tag for word, tag in nltk.pos_tag(tokens)]

        rows.append({
            "Character": name,
            "Vocabulary size": len(vocabulary),
            "Tokens": len(tokens),
            "Adjectives": sum(t.startswith("JJ") for t in tags),
            "Adverbs": sum(t.startswith("RB") for t in tags),
            "Verbs": sum(t.startswith("VB") for t in tags),
            "Nouns": sum(t.startswith("NN") for t in tags),
            "Vocabulary (sample)": sorted(vocabulary)[:5],
        })

    summary = pd.DataFrame(rows).sort_values("Tokens", ascending=False)
    print(summary.to_string(index=False))


In [ ]:
task2()


# Task 3

In [ ]:
def task3():
    speech = getSpeech()
    plt.gca().set_prop_cycle(color=plt.cm.tab20.colors)

    rows = []
    for name, quotes in speech.items():
        df = pd.DataFrame({"text": quotes})
        tokens = nltk.word_tokenize(" ".join(df["text"]))
        words = [w.lower() for w in tokens if w.isalpha()]

        # vocabulary size after each token
        N, V, seen = [], [], set()
        for i, w in enumerate(words, 1):
            seen.add(w)
            N.append(i)
            V.append(len(seen))
        plt.plot(N, V, label=name)

        # Heaps' law
        x, y = np.log(N), np.log(V)
        beta, logK = np.polyfit(x, y, 1)
        y_pred = logK + beta * x
        r2 = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2)
        n = len(x)
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - 2)

        rows.append({
            "Character": name,
            "Tokens": len(words),
            "Vocabulary": len(seen),
            "K": round(np.exp(logK), 2),
            "beta": round(beta, 3),
            "R2": round(r2, 4),
            "Adj. R2": round(adj_r2, 4),
        })

    plt.xlabel("Number of tokens")
    plt.ylabel("Vocabulary size")
    plt.title("Vocabulary growth per character")
    plt.legend()
    plt.show()

    summary = pd.DataFrame(rows).sort_values("Tokens", ascending=False)
    print(summary.to_string(index=False))


In [ ]:
task3()


# Task 4

In [ ]:
def task4():
    modalVerbs = ("can", "may", "must", "shall", "will",
                  "could", "might", "should", "would")

    speechAsTokens = getSpeechAsTokens()

    # todo: normalize graph?
    displayRaw = False
    for person, tokens in speechAsTokens.items():
        xEntries = []
        yEntries = []
        tokenCount = 0
        modalVerbCount = 0

        for token in tokens:
            tokenCount += 1
            if token in modalVerbs:
                modalVerbCount += 1
            xEntries.append(tokenCount)
            yEntries.append(modalVerbCount)

        # Calculate fitted values
        a, b = np.polyfit(xEntries, yEntries, 1)
        yEntriesFitted = []
        for xEntry in xEntries:
            yFitted = a * xEntry + b
            yEntriesFitted.append(yFitted)

        if displayRaw:
            # Draw raw data
            plt.plot(xEntries, yEntries, label=person)
        else:
            # Draw fitted line
            plt.plot(xEntries, yEntriesFitted, label=person)
            # Draw confidence interval
            xEntries = np.array(xEntries)
            yEntries = np.array(yEntries)
            xEntriesConst = sm.add_constant(xEntries)
            model = sm.OLS(yEntries, xEntriesConst).fit()
            xPredictions = np.linspace(xEntries.min(), xEntries.max())
            xPredictionsConst = sm.add_constant(xPredictions)
            pred = model.get_prediction(
                xPredictionsConst).summary_frame(alpha=0.10)
            plt.fill_between(
                xPredictions,
                pred["obs_ci_lower"],
                pred["obs_ci_upper"],
                alpha=0.5,
            )

    plt.xlabel("Number of tokens")
    plt.ylabel("Number of modal verbs")
    plt.title("Modal verbs compared to total tokens for each character")
    plt.legend()
    plt.show()


In [ ]:
task4()


# Task 5

In [ ]:
def task5():
    tokens = getBookTokens()
    filteredTokens = filterTokenListTask5And6(tokens)

    counter = Counter(filteredTokens)
    mostCommon = counter.most_common(5)
    tokenArray, countArray = zip(*mostCommon)
    print("Most frequent tokens in the whole textbook:")
    printMatrix([tokenArray, countArray], labels=False)
    print()

    mostCommonVector = []
    for token, count in mostCommon:
        mostCommonVector.append(token)

    vectorDict = dict()
    speechAsTokens = getSpeechAsTokens()
    for person, tokens in speechAsTokens.items():
        vectorDict[person] = [0, 0, 0, 0, 0]
        for token in tokens:
            for i, mostCommonToken in enumerate(mostCommonVector):
                if token == mostCommonToken:
                    vectorDict[person][i] += 1

    frequencyMatrix = list(vectorDict.values())

    print("Frequency of the five most frequent tokens for each character:")
    printMatrix(frequencyMatrix,
                rowLabels=vectorDict.keys(), columnLabels=mostCommonVector)
    print()

    cosineSimilarityMatrix = cosine_similarity(
        frequencyMatrix, frequencyMatrix)

    print("Cosine similarity between characters")
    printMatrix(cosineSimilarityMatrix, rowLabels=vectorDict.keys(),
                columnLabels=vectorDict.keys())


In [ ]:
task5()


# Task 6

In [ ]:
def task6():
    speechAsFilteredTokens = getSpeechAsFilteredTokens(filterTokenListTask5And6)
    people = speechAsFilteredTokens.keys()
    filteredTokenLists = list(speechAsFilteredTokens.values())

    dictionary = corpora.Dictionary(filteredTokenLists)
    corpus = [dictionary.doc2bow(text) for text in filteredTokenLists]

    topicCount = 5
    keywordCount = 10

    ldamodel = gensim.models.ldamodel.LdaModel(
        corpus, id2word=dictionary, num_topics=topicCount)
    topicStrings = ldamodel.print_topics(num_words=keywordCount)

    print("Topics:")
    for topic in topicStrings:
        print(topic)
    print()

    topics = ldamodel.show_topics(
        num_topics=ldamodel.num_topics,
        num_words=keywordCount,
        formatted=False
    )

    personTopicSimilarities = dict()
    for person in people:
        counter = Counter(speechAsFilteredTokens[person])

        personTopicSimilarities[person] = [0, 0, 0, 0, 0]
        for topicId, words in topics:
            for word, probability in words:
                if word in speechAsFilteredTokens[person]:
                    weight = probability * counter[word]
                    personTopicSimilarities[person][topicId] += weight

    print("Each person's similarity with each topic:")
    printMatrixTransposed(personTopicSimilarities)
    print()

    personTopicSimilarityMatrix = list(personTopicSimilarities.values())
    cosineSimilarityMatrix = cosine_similarity(
        personTopicSimilarityMatrix, personTopicSimilarityMatrix)

    print("Topic similarity between characters using cosine similarity")
    printMatrix(cosineSimilarityMatrix, columnLabels=people, rowLabels=people)


In [ ]:
task6()


# Task 7

In [ ]:
def task7():
    speechAsFilteredTokens = getSpeechAsFilteredTokens(filterTokenListTask7)
    people = speechAsFilteredTokens.keys()
    filteredTokenLists = list(speechAsFilteredTokens.values())

    topicCount = 5
    keywordCount = 10
    load_dotenv()
    model = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
    # Prompt generated with help of ChatGPT
    prompt = (
        f"Find {topicCount} shared topics in these dialogue collections. "
        f"For each topic choose {keywordCount} keywords from the supplied tokens. "
        "Assign positive keyword weights summing to one within each topic. "
        'Return JSON as {"topics": [[0, [["word", 0.1], ...]], ...]}. '
        "Use topic IDs 0 to 4.\n" + json.dumps(filteredTokenLists)
    )
    # https://ai.google.dev/api/generate-content#method:-models.generatecontent
    response = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent",
        headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]},
        json={
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": 0, "responseMimeType": "application/json"},
        },
        timeout=180,
    )
    text = response.json()["candidates"][0]["content"]["parts"][-1]["text"]
    topics = json.loads(text)["topics"]
    print("Topics:")
    for topic in topics:
        print(topic)
    print()

    personTopicSimilarities = dict()
    for person in people:
        counter = Counter(speechAsFilteredTokens[person])

        personTopicSimilarities[person] = [0, 0, 0, 0, 0]
        for topicId, words in topics:
            for word, probability in words:
                if word in speechAsFilteredTokens[person]:
                    weight = probability * counter[word]
                    personTopicSimilarities[person][topicId] += weight

    print("Each person's similarity with each topic:")
    printMatrix(personTopicSimilarities)
    print()

    personTopicSimilarityMatrix = list(personTopicSimilarities.values())
    cosineSimilarityMatrix = cosine_similarity(
        personTopicSimilarityMatrix, personTopicSimilarityMatrix)

    print("Topic similarity between characters using cosine similarity")
    printMatrix(cosineSimilarityMatrix, columnLabels=people, rowLabels=people)


In [ ]:
task7()


# Task 8

In [ ]:
def getBertVector(text, tokenizer, model):
    inputs = tokenizer(text, truncation=True, return_overflowing_tokens=True)
    total = torch.zeros(model.config.hidden_size)
    count = 0

    for ids in inputs["input_ids"]:
        input_ids = torch.tensor([ids])
        with torch.no_grad():
            result = model(input_ids=input_ids)
            token_vectors = result.last_hidden_state
            token_vectors = token_vectors[0]
            output = token_vectors[1:-1]
        total += output.sum(dim=0)
        count += len(output)

    return (total / count).numpy()


def task8(dataframes):
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased")

    vectors = []
    for _name, df in dataframes.items():
        vector = getBertVector(" ".join(df["text"]), tokenizer, model)
        vectors.append(vector)

    names = list(dataframes)
    embeddings = pd.DataFrame(vectors, index=names)
    similarity = pd.DataFrame(cosine_similarity(vectors), index=names, columns=names)
    print(f"BERT similarity:\n{similarity.round(3).to_string()}")
    return embeddings, similarity


In [ ]:
task8(dataframes)


# Task 9

In [ ]:
def task9():
    lexicon = Empath()

    empathDict = dict()
    vectorDict = dict()
    speech = getSpeech()
    for person, text in speech.items():
        empathDict[person] = lexicon.analyze(text, normalize=False)
        vectorDict[person] = list(empathDict[person].values())

    print(empathDict['Elizabeth'])

    personEmpathMatrix = list(vectorDict.values())
    cosineSimilarityMatrix = cosine_similarity(
        personEmpathMatrix, personEmpathMatrix)

    people = speech.keys()

    print("Similarity between characters using empath categories")
    printMatrix(cosineSimilarityMatrix, columnLabels=people, rowLabels=people)


In [ ]:
task9()


# Task 10

In [ ]:
def getSentiment(text, lemmatizer):
    tokens = []
    for token in nltk.word_tokenize(text):
        for character in token:
            if character.isalnum():
                tokens.append(token)
                break

    tagged = nltk.pos_tag(tokens)
    positive, negative, neutral = 0.0, 0.0, 0.0
    pos_tags = {"J": "a", "N": "n", "R": "r", "V": "v"}

    for word, tag in tagged:
        pos = pos_tags.get(tag[0])
        if pos is None:
            neutral += 1
            continue
        lemma = lemmatizer.lemmatize(word.lower(), pos)
        synset = lesk(tokens, lemma, pos)
        if synset is not None:
            sentiment = swn.senti_synset(synset.name())
            positive += sentiment.pos_score()
            negative += sentiment.neg_score()
            neutral += sentiment.obj_score()
        else:
            neutral += 1

    if not tokens:
        return [0.0, 0.0, 1.0]
    return [positive / len(tokens), negative / len(tokens), neutral / len(tokens)]


def task10(dataframes):
    lemmatizer = WordNetLemmatizer()
    results = []
    rows = ceil(len(dataframes) / 3)
    fig = plt.figure(figsize=(12, rows * 3.5), layout="constrained")

    for i, (name, df) in enumerate(dataframes.items(), 1):
        scores = []
        for message, text in enumerate(df["text"], 1):
            positive, negative, neutral = getSentiment(text, lemmatizer)
            scores.append([positive, negative, neutral])
            results.append([name, message, text, positive, negative, neutral])

        scores = pd.DataFrame(scores, columns=["Positive", "Negative", "Neutral"])
        ax = fig.add_subplot(rows, 3, i, projection="ternary")
        ax.scatter(scores["Positive"], scores["Negative"], scores["Neutral"], s=12, alpha=0.5)
        ax.set_tlabel("Positive")
        ax.set_llabel("Negative")
        ax.set_rlabel("Neutral")
        ax.set_title(name)
        ax.grid(alpha=0.3)

    results = pd.DataFrame(results, columns=["Character", "Message", "Text", "Positive", "Negative", "Neutral"])
    print(f"Sentiment scores:\n{results.drop(columns='Text').round(3)}")
    plt.show()
    return results


In [ ]:
task10(dataframes)
